# DroneVehicle RGB/IR 配准质量审计

## tl;dr

数据集的 RGB/IR 文件配对、分辨率和共享标签均完整一致；几何上属于较好配准，但不能把同一车辆严格视为逐像素完全相同的 `(x, y)`。可信跨模态配准样本的残余平移中位数约 1.74 px，仍有一部分序列达到 5–10 px。

## Context & Methods

审计分两层：一是对全部 train/val/test 文件检查同名配对、可读性与尺寸；二是对400 对均匀抽样图像，用梯度幅值上的相位相关和 translation-only ECC 估计全局残余位移，并对其中 40 对追加仿射 ECC。标签框尺寸用所有多边形的最小外接旋转矩形计算。

### Key Assumptions

- ECC 相关系数至少 0.50 且位移不超过 10 px 的结果记为可信。
- 该估计衡量场景级几何关系，不是每辆车的独立双模态中心标注。
- 热辐射、阴影和运动目标会降低跨模态相关性。

In [1]:
from pathlib import Path
import csv, json, subprocess, sys
from IPython.display import Markdown, display

REPORT_DIR = Path.cwd()
if not (REPORT_DIR / 'analyze_alignment.py').is_file():
    REPORT_DIR = REPORT_DIR / 'reports/dronevehicle_alignment_quality'
# Re-check every pair and label while reusing the already generated deterministic ECC sample.
# Remove --reuse-alignment to regenerate the expensive cross-modal registration estimates.
subprocess.run([sys.executable, str(REPORT_DIR / 'analyze_alignment.py'), '--reuse-alignment'], check=True)
summary = json.loads((REPORT_DIR / 'summary.json').read_text(encoding='utf-8'))
summary['total_exact_triplets'], summary['alignment_sample_count']

(28439, 400)

## Data integrity

In [2]:
display({
    'columns': ['split', 'rgb_files', 'ir_files', 'label_files', 'triplet_coverage_pct', 'dimension_match_pct'],
    'rows': [[r[k] for k in ['split', 'rgb_files', 'ir_files', 'label_files', 'triplet_coverage_pct', 'dimension_match_pct']] for r in summary['pairing']]
})
display(Markdown(f"**Valid polygon rows:** {summary['box_instances']:,}; malformed rows: {summary['invalid_label_rows']}."))

{'columns': ['split',
  'rgb_files',
  'ir_files',
  'label_files',
  'triplet_coverage_pct',
  'dimension_match_pct'],
 'rows': [['train', 17990, 17990, 17990, 100.0, 100.0],
  ['val', 1469, 1469, 1469, 100.0, 100.0],
  ['test', 8980, 8980, 8980, 100.0, 100.0]]}

**Valid polygon rows:** 500,515; malformed rows: 0.

## Geometric alignment results

In [3]:
lines = [
    f"**Trusted estimates:** {summary['trusted_alignment_count']}/{summary['alignment_sample_count']} ({summary['trusted_alignment_rate_pct']:.2f}%).",
    f"**Residual translation:** median {summary['median_shift_px']:.3f} px; p90 {summary['p90_shift_px']:.3f} px.",
    f"**Within 2 px:** {summary['share_within_2px_pct']:.2f}%; within 3 px: {summary['share_within_3px_pct']:.2f}%.",
    f"**Affine check:** median center shift {summary['affine_median_center_shift_px']:.3f} px; p90 maximum corner shift {summary['affine_p90_max_corner_shift_px']:.3f} px.",
    f"**Object scale:** median target short side {summary['box_size_summary'][0]['short_side_p50_px']:.2f} px; median shift is {summary['median_shift_as_pct_of_median_short_side']:.2f}% of it.",
]
display(Markdown('\n\n'.join(lines)))
display({'columns': list(summary['split_alignment_summary'][0]), 'rows': [list(r.values()) for r in summary['split_alignment_summary']]})

**Trusted estimates:** 293/400 (73.25%).

**Residual translation:** median 1.741 px; p90 6.857 px.

**Within 2 px:** 54.27%; within 3 px: 63.82%.

**Affine check:** median center shift 1.709 px; p90 maximum corner shift 10.451 px.

**Object scale:** median target short side 25.00 px; median shift is 6.97% of it.

{'columns': ['split',
  'samples',
  'trusted_samples',
  'trusted_rate_pct',
  'median_shift_px',
  'p90_shift_px',
  'within_2px_pct'],
 'rows': [['train', 200, 156, 78.0, 1.719, 6.513, 53.85],
  ['val', 50, 33, 66.0, 3.008, 8.081, 48.48],
  ['test', 150, 104, 69.33, 1.719, 5.786, 56.73]]}

## Takeaways

- **近似成立，严格不成立。** 对多数可信样本，可把同一车辆在两模态中的中心理解为相差约 1–3 px，而不是数学上完全相同的坐标。
- **存在序列性例外。** 约 18.8% 的可信估计落在 5–10 px；高位移可视化中道路线、灯杆和车辆共同出现方向一致的红/绿边缘，说明并非只有车辆热轮廓变化。
- **对小目标不可忽略。** 全体目标短边中位数约 25 px，1.74 px 已占约 7%；5–10 px 对小车的逐像素融合会很明显。
- 共享一套 OBB 标签只说明训练管线把两模态当作同坐标监督，不证明两幅原始图像逐像素对齐。